# FlyWire Quick Test: Synapse Count Measurement Error

**2 rates × 1 seed = 2 trials — fast validation run.**

For EM3, error_rate means relative measurement uncertainty on each edge weight.

In [ ]:
# Cell 1: Environment Setup
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists('/kaggle/input')
KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    sys.path.insert(0, str(KAGGLE_CODEBASE_PATH))
    print(f'[OK] Codebase: {KAGGLE_CODEBASE_PATH}')
else:
    REPO = Path(os.getcwd())
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    print(f'[OK] Local: {REPO}')

In [ ]:
# Cell 2: Imports
from core.experiment_runner import ExperimentRunner, ExperimentConfig
from modules.error_models import registry as error_registry
from modules.graph_analyses.analysis_registry import registry as analysis_registry
from modules.statistical_evaluation import StatisticalEvaluator
from core.export_manager import ExportManager
print('Imports OK')

In [ ]:
# ============================================================
# Cell 3: MINIMAL TEST CONFIG  (2 rates, 1 seed)
# ============================================================
DATASET_NAME = "BANC"
LOCAL_DATASET_ROOT = "research_data/raw"

EXPERIMENT = {
    "metadata": {
        "experiment_name": f"BANC_SynapseCount_{DATASET_NAME}",
        "author": "FlyWire Researcher",
        "description": "QUICK TEST — synapse count measurement on BANC (2 rates, 1 seed).",
    },
    "error": {
        "name": "synapse_count_measurement",
        "rates": [
            0.00,    # baseline (required)
            0.05,    # 5% — one perturbed rate
        ],
        "random_seeds": [1],
    },
    "analysis": [
        "basic_structure",
        "degree_distribution",
        "pagerank",
        "assortativity",
        "connected_components",
        "reciprocity",
    ],
    "export": {
        "create_zip": True,
        "save_statistics": True,
    },
}

OUTPUT_ROOT = Path("results") / DATASET_NAME / EXPERIMENT["error"]["name"]
print(f'Config: {DATASET_NAME} | rates={len(EXPERIMENT["error"]["rates"])} | seeds={len(EXPERIMENT["error"]["random_seeds"])}')

In [ ]:
# Cell 4: Dataset Root
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT
CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs') if IS_KAGGLE else 'configs'
print(f'DATASET_ROOT = {DATASET_ROOT}')

In [ ]:
# Cell 5: Verify Dataset
from core.dataset_registry import DatasetRegistry
reg = DatasetRegistry(configs_root=CONFIGS_ROOT, dataset_root=DATASET_ROOT)
_ = reg.resolve_dataset_dir(DATASET_NAME, DATASET_ROOT)
print(f'[OK] Dataset "{DATASET_NAME}" resolved.')

In [ ]:
# Cell 6: Verify Registries
err_model = EXPERIMENT['error']['name']
print(f'Models: {error_registry.list_names()}')
print(f'Analyses: {analysis_registry.list_names()}')
assert err_model in error_registry.list_names(), f'{err_model} not registered!'
print('[OK] Ready.')

In [ ]:
# Cell 7: Run Experiments (2 rates x 1 seed = 2 trials)
import time
t_start = time.perf_counter()

runner = ExperimentRunner(analysis_registry, error_registry)
results_per_rate = {}

for err_rate in EXPERIMENT['error']['rates']:
    rate_str = f"{int(err_rate*100)}_percent"
    results_per_rate[err_rate] = []
    for trial, seed in enumerate(EXPERIMENT['error']['random_seeds'], 1):
        print(f'[{rate_str} | trial {trial}] seed={seed} ...', end=' ')
        trial_out = OUTPUT_ROOT / rate_str / f'trial_{trial:03d}'
        config = ExperimentConfig(
            dataset_name=DATASET_NAME,
            dataset_root=str(DATASET_ROOT),
            configs_root=CONFIGS_ROOT,
            error_model_name=err_model,
            error_model_config={'error_rate': err_rate},
            analysis_names=EXPERIMENT['analysis'],
            preprocessing_config={'features': {'degree': True, 'synapse_counts': True}},
            seed=seed,
            output_root=str(trial_out) if EXPERIMENT['export']['save_statistics'] else None,
            create_zip=EXPERIMENT['export']['create_zip'],
            extra={'metadata': EXPERIMENT['metadata']},
        )
        res = runner.run(config)
        results_per_rate[err_rate].append(res)
        status = 'OK' if res.succeeded else 'FAIL'
        meta = res.error_result.perturbation_metadata if res.error_result else {}
        pct = meta.get('pct_edges_changed', 0)
        print(f'{status} ({res.runtime_seconds:.2f}s, {pct:.1f}% edges changed)')

print(f'\nAll trials done in {time.perf_counter()-t_start:.1f}s')

In [ ]:
# Cell 8: Statistical Evaluation
evaluator = StatisticalEvaluator()
aggregated_stats_by_rate = {}
baseline_runs = [r for r in results_per_rate.get(0.00, []) if r.succeeded]
print(f'Baseline runs: {len(baseline_runs)}')

for err_rate, run_results in results_per_rate.items():
    successful = [r for r in run_results if r.succeeded]
    if successful and err_rate > 0:
        eval_result = evaluator.evaluate(baseline_runs, successful)
        aggregated_stats_by_rate[err_rate] = eval_result
        print(f'  {err_rate*100:g}%: {len(successful)} trials evaluated')

print('Evaluation complete.')

In [ ]:
# Cell 9: Export Presentation
ExportManager().export_presentation(
    results_by_rate=aggregated_stats_by_rate,
    output_root=OUTPUT_ROOT,
    metadata=EXPERIMENT['metadata'],
)
print(f'Presentation -> {OUTPUT_ROOT / "presentation"}')

In [ ]:
# Cell 10: Quick Summary
print('=' * 50)
print('  BANC QUICK TEST - SYNAPSE COUNT MEASUREMENT')
print('=' * 50)
for err_rate in sorted(aggregated_stats_by_rate.keys()):
    ev = aggregated_stats_by_rate[err_rate]
    print(f'  Uncertainty {err_rate*100:g}%')
    for a_name, metrics in ev.metrics.items():
        print(f'    {a_name}: {len(metrics)} metrics')
        for m_name, m_dict in list(metrics.items())[:3]:
            print(f'      {m_name}: mean={m_dict.mean:.4f} d={m_dict.effect_size:.4f}')
total_trials = sum(len(v) for v in results_per_rate.values())
print(f'  Trials: {total_trials} | Rates: {len(aggregated_stats_by_rate)}')
print(f'  Output: {OUTPUT_ROOT}')
print('=' * 50)